In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')


In [2]:
df=pd.read_csv('train.txt',sep=';',header=None,names=['text','emotion'])
df.head()

,text,emotion
0,i didnt feel humiliated,sadness
1,i can go from feeling so hopeless to so damned...,sadness
2,im grabbing a minute to post i feel greedy wrong,anger
3,i am ever feeling nostalgic about the fireplac...,love
4,i am feeling grouchy,anger


In [3]:
df.isnull().sum()

text       0
emotion    0
dtype: int64

In [4]:
unique_emotions=df['emotion'].unique()
emotion_numbers={}
i=0
for emo in unique_emotions:
    emotion_numbers[emo]=i
    i+=1
df['emotion']=df['emotion'].map(emotion_numbers)

In [5]:
df

,text,emotion
0,i didnt feel humiliated,0
1,i can go from feeling so hopeless to so damned...,0
2,im grabbing a minute to post i feel greedy wrong,1
3,i am ever feeling nostalgic about the fireplac...,2
4,i am feeling grouchy,1
...,...,...
15995,i just had a very brief time in the beanbag an...,0
15996,i am now turning and i feel pathetic that i am...,0
15997,i feel strong and good overall,5
15998,i feel like this was such a rude comment and i...,1


In [6]:
df['text']=df['text'].apply(lambda x:x.lower())

In [7]:
import string
def remove_punctuation(text):
    for punctuation in string.punctuation:
        text=text.replace(punctuation,'')
    return text
df['text']=df['text'].apply(remove_punctuation)

In [8]:
df

,text,emotion
0,i didnt feel humiliated,0
1,i can go from feeling so hopeless to so damned...,0
2,im grabbing a minute to post i feel greedy wrong,1
3,i am ever feeling nostalgic about the fireplac...,2
4,i am feeling grouchy,1
...,...,...
15995,i just had a very brief time in the beanbag an...,0
15996,i am now turning and i feel pathetic that i am...,0
15997,i feel strong and good overall,5
15998,i feel like this was such a rude comment and i...,1


In [9]:
def remove_numbers(txt):
  new=""
  for i in txt:
    if not i.isdigit():
      new=new+i
  return new

In [10]:
df['text']=df['text'].apply(remove_numbers)

In [11]:
import re

def remove_emojis(text):
    # Remove characters matching emoji ranges in Unicode
    return text.encode('ascii', 'ignore').decode('ascii')

df['text'] = df['text'].apply(remove_emojis)
df.head()

,text,emotion
0,i didnt feel humiliated,0
1,i can go from feeling so hopeless to so damned...,0
2,im grabbing a minute to post i feel greedy wrong,1
3,i am ever feeling nostalgic about the fireplac...,2
4,i am feeling grouchy,1


In [12]:
import os
import nltk
import shutil

# Use a clean NLTK directory
nltk_dir = os.path.join(os.path.expanduser("~"), "nltk_data")

# Remove the conflicting old stopwords installation
old_stopwords = os.path.join(nltk_dir, "corpora", "stopwords")

if os.path.exists(old_stopwords):
    shutil.rmtree(old_stopwords)

# Configure NLTK
nltk.data.path = [nltk_dir]

# Download stopwords again
nltk.download("stopwords", download_dir=nltk_dir, force=True)

# Verify
from nltk.corpus import stopwords

stop_words = set(stopwords.words("english"))

print("Successfully loaded stopwords!")
print("Total:", len(stop_words))

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Akshat\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.


Successfully loaded stopwords!
Total: 198


In [13]:
import nltk
import os

print("NLTK version:", nltk.__version__)
print("NLTK paths:", nltk.data.path)
print("NLTK_DATA environment:", os.environ.get("NLTK_DATA"))

print("Stopwords locations:")
for path in nltk.data.path:
    print(path, os.path.exists(os.path.join(path, "corpora", "stopwords")))

NLTK version: 3.10.3
NLTK paths: ['C:\\Users\\Akshat\\nltk_data']
NLTK_DATA environment: None
Stopwords locations:
C:\Users\Akshat\nltk_data True


In [14]:
from nltk.corpus import stopwords

stop_words = set(stopwords.words("english"))

print("Stopwords loaded successfully!")
print("Total stopwords:", len(stop_words))
print("Sample:", list(stop_words)[:10])

Stopwords loaded successfully!
Total stopwords: 198
Sample: ['over', 'there', 'ours', 'too', 'it', "she'll", 'than', 'other', 'y', "wouldn't"]


In [15]:
df['text'] = df['text'].apply(
    lambda text: " ".join(
        word for word in text.split()
        if word not in stop_words
    )
)

In [16]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(df['text'],df['emotion'],test_size=0.2,random_state=42)

In [17]:
from sklearn.feature_extraction.text import TfidfVectorizer,CountVectorizer

In [18]:
bow_vectorizer=CountVectorizer()
X_train_bow=bow_vectorizer.fit_transform(X_train)
X_test_bow=bow_vectorizer.transform(X_test)

In [19]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score,classification_report

In [20]:
nv_model=MultinomialNB()
nv_model.fit(X_train_bow,y_train)
y_pred=nv_model.predict(X_test_bow)

In [21]:
accuracy_score(y_test,y_pred)

0.768125

In [22]:
tfidf_vectorizer=TfidfVectorizer()
X_train_tfidf=tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf=tfidf_vectorizer.transform(X_test)


nb2_model=MultinomialNB()
nb2_model.fit(X_train_tfidf,y_train)
y_pred2=nb2_model.predict(X_test_tfidf)

In [23]:
accuracy_score(y_test,y_pred2)


0.6609375

In [24]:
from sklearn.linear_model import LogisticRegression
lr_model=LogisticRegression(max_iter=1000)
lr_model.fit(X_train_tfidf,y_train)
y_pred3=lr_model.predict(X_test_tfidf)

In [25]:
accuracy_score(y_test,y_pred3)

0.8628125

In [ ]:
%pip install joblib

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 26.1.2 -> 26.3
[notice] To update, run: C:\Users\Akshat\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [27]:
import joblib
import os

In [28]:
# Create a folder to store model files
os.makedirs("models", exist_ok=True)

# Save trained model
joblib.dump(lr_model, "models/emotion_model.pkl")

# Save TF-IDF vectorizer
joblib.dump(tfidf_vectorizer, "models/tfidf_vectorizer.pkl")

# Save emotion label mapping
joblib.dump(emotion_numbers, "models/emotion_mapping.pkl")

print("All model files saved successfully!")

All model files saved successfully!


In [29]:
print(os.listdir("models"))

['emotion_mapping.pkl', 'emotion_model.pkl', 'tfidf_vectorizer.pkl']


In [30]:
model = joblib.load("models/emotion_model.pkl")
vectorizer = joblib.load("models/tfidf_vectorizer.pkl")
emotion_mapping = joblib.load("models/emotion_mapping.pkl")

print("All files loaded successfully!")

All files loaded successfully!
